In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("../data/processed/used_cars_clean.csv")

In [6]:
CURRENT_YEAR = 2026

df["vehicle_age"] = CURRENT_YEAR - df["year"]

In [7]:
df[["year", "vehicle_age"]].head()

,year,vehicle_age
0,2019,7
1,2020,6
2,2016,10
3,2020,6
4,2020,6


In [8]:
df["mileage_per_year"] = (
    df["mileage"] /
    df["vehicle_age"].replace(0, 1)
)

In [9]:
df[[
    "year",
    "mileage",
    "vehicle_age",
    "mileage_per_year"
]].head()

,year,mileage,vehicle_age,mileage_per_year
0,2019,7.0,7,1.000000
1,2020,8.0,6,1.333333
2,2016,8267.0,10,826.700000
3,2020,11.0,6,1.833333
4,2020,7.0,6,1.166667


In [11]:
df["is_large_engine"] = (
    df["engine_displacement"] >= 3000
)

In [12]:
df[["engine_displacement", "is_large_engine"]].head(10)

,engine_displacement,is_large_engine
0,1300.0,False
1,2000.0,False
2,2500.0,False
3,3000.0,True
4,2000.0,False
5,2000.0,False
6,2500.0,False
7,2000.0,False
8,2000.0,False
9,2000.0,False


In [13]:
df["accident_status"] = df["has_accidents"]

In [14]:
df["is_efficient"] = (
    df["highway_fuel_economy"] >= 30
)

In [15]:
df.columns

Index(['price', 'make_name', 'model_name', 'year', 'mileage', 'body_type',
       'fuel_type', 'transmission', 'engine_displacement', 'engine_cylinders',
       'engine_type', 'wheel_system', 'has_accidents', 'frame_damaged',
       'city_fuel_economy', 'highway_fuel_economy', 'vehicle_age',
       'mileage_per_year', 'is_large_engine', 'accident_status',
       'is_efficient'],
      dtype='str')

# 03 Feature Engineering

## 생성한 Feature

### vehicle_age
현재 연도 기준 차량의 나이

### mileage_per_year
연평균 주행거리

### is_large_engine
배기량 3000cc 이상 여부

### accident_status
사고 여부

### is_efficient
고속도로 연비 30MPG 이상 여부

이 프로젝트는 단순한 과제가 아니라 포트폴리오입니다.

그래서 새로운 Feature를 만들 때는 항상 "왜 만들었는지"를 설명할 수 있어야 합니다.

예를 들어:

vehicle_age: 같은 주행거리라도 연식이 다르면 차량 상태에 대한 해석이 달라질 수 있기 때문.
mileage_per_year: 총 주행거리보다 차량 사용 강도를 더 잘 나타낼 수 있기 때문.
is_large_engine: 배기량은 차량 성능과 가격에 영향을 줄 수 있기 때문.

🚀 그런데 여기서 제가 하나 업그레이드 제안을 드리고 싶습니다.

지금까지는 기본 Feature Engineering입니다.

하지만 이 데이터는 300만 건이라 더 재미있는 특징도 만들 수 있습니다.

예를 들면:

차종별 평균 가격과의 차이
브랜드별 평균 가격
모델별 평균 주행거리
브랜드의 프리미엄 여부(럭셔리 브랜드인지)

이런 특징들은 성능을 크게 높일 수 있지만, 데이터 누수(Data Leakage)가 생기지 않도록 주의해서 만들어야 합니다.

제 추천은
지금은 위의 기본 Feature 5개만 생성합니다.
그다음 바로 CatBoost 첫 번째 모델을 학습해서 기준 성능(Baseline)을 만듭니다.
이후 더 고급 Feature를 하나씩 추가하면서 성능이 얼마나 좋아지는지 비교해봅시다.

이렇게 하면 "Feature Engineering이 실제로 모델 성능을 얼마나 개선했는지"까지 보여줄 수 있는, 훨씬 설득력 있는 포트폴리오가 됩니다.

In [16]:
df["mileage_level"] = pd.cut(
    df["mileage_per_year"],
    bins=[0, 5000, 10000, 15000, 1000000],
    labels=["Low", "Normal", "High", "Very High"]
)

연평균 주행거리	등급
3,000	Low
8,000	Normal
13,000	High
25,000	Very High

이런 식으로 차량 사용 강도를 범주형으로 표현할 수 있습니다.

In [17]:
df.to_csv(
    "../data/processed/used_cars_featured.csv",
    index=False
)

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000040 entries, 0 to 3000039
Data columns (total 22 columns):
 #   Column                Dtype   
---  ------                -----   
 0   price                 float64 
 1   make_name             str     
 2   model_name            str     
 3   year                  int64   
 4   mileage               float64 
 5   body_type             str     
 6   fuel_type             str     
 7   transmission          str     
 8   engine_displacement   float64 
 9   engine_cylinders      str     
 10  engine_type           str     
 11  wheel_system          str     
 12  has_accidents         str     
 13  frame_damaged         str     
 14  city_fuel_economy     float64 
 15  highway_fuel_economy  float64 
 16  vehicle_age           int64   
 17  mileage_per_year      float64 
 18  is_large_engine       bool    
 19  accident_status       str     
 20  is_efficient          bool    
 21  mileage_level         category
dtypes: bool(2), category(1), floa

In [19]:
df.head()

,price,make_name,model_name,year,mileage,body_type,fuel_type,transmission,engine_displacement,engine_cylinders,...,has_accidents,frame_damaged,city_fuel_economy,highway_fuel_economy,vehicle_age,mileage_per_year,is_large_engine,accident_status,is_efficient,mileage_level
0,23141.0,Jeep,Renegade,2019,7.0,SUV / Crossover,Gasoline,A,1300.0,I4,...,Unknown,Unknown,21.0,29.0,7,1.000000,False,Unknown,False,Low
1,46500.0,Land Rover,Discovery Sport,2020,8.0,SUV / Crossover,Gasoline,A,2000.0,I4,...,Unknown,Unknown,21.0,29.0,6,1.333333,False,Unknown,False,Low
2,46995.0,Subaru,WRX STI,2016,8267.0,Sedan,Gasoline,M,2500.0,H4,...,False,False,17.0,23.0,10,826.700000,False,False,False,Low
3,67430.0,Land Rover,Discovery,2020,11.0,SUV / Crossover,Gasoline,A,3000.0,V6,...,Unknown,Unknown,21.0,29.0,6,1.833333,True,Unknown,False,Low
4,48880.0,Land Rover,Discovery Sport,2020,7.0,SUV / Crossover,Gasoline,A,2000.0,I4,...,Unknown,Unknown,21.0,29.0,6,1.166667,False,Unknown,False,Low


In [20]:
df.columns

Index(['price', 'make_name', 'model_name', 'year', 'mileage', 'body_type',
       'fuel_type', 'transmission', 'engine_displacement', 'engine_cylinders',
       'engine_type', 'wheel_system', 'has_accidents', 'frame_damaged',
       'city_fuel_economy', 'highway_fuel_economy', 'vehicle_age',
       'mileage_per_year', 'is_large_engine', 'accident_status',
       'is_efficient', 'mileage_level'],
      dtype='str')